# Numerical & Data Libraries for Python
### NumPy · Pandas · Matplotlib — Hands-on primer

Welcome! This is a first-exposure tour of the three libraries almost every data task in Python relies on. Each idea is one short cell — **run them top-to-bottom and watch the output.**

**Running example:** a snapshot of five lending peers — `BAJFINANCE`, `HDFCBANK`, `KOTAKBANK`, `SBIN`, `ICICIBANK`. We meet them as a price array (NumPy), grow them into a table (Pandas), then chart them (Matplotlib). All figures here are *illustrative*, not live market data.

> **Tip:** click a cell and press **Shift + Enter** to run it.

## 0 · Setup

One import block for the whole session. In modern Jupyter / VS Code the inline backend is already the default, so `%matplotlib inline` is optional — included only as a habit.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

%matplotlib inline
print("numpy", np.__version__, "| pandas", pd.__version__)

---
# Part 1 · NumPy — Numerical Computing

NumPy gives us the `ndarray`: a fast, fixed-type, n-dimensional array. Think of it as the engine under Pandas. We start with six month-end closing prices for `BAJFINANCE` (illustrative).

### 1.1 · Arrays & their properties

In [ ]:
prices = np.array([6800, 6950, 7100, 6900, 7050, 7200.0])   # BAJFINANCE month-end (illustrative)
prices

In [ ]:
# Four properties you will reach for constantly
prices.shape, prices.dtype, prices.ndim, prices.size

### 1.2 · Creating arrays

You rarely type arrays by hand — you generate them.

In [ ]:
np.zeros(3), np.ones(3), np.arange(0, 6)

In [ ]:
np.linspace(0, 1, 5)        # 5 evenly spaced points, endpoints included

In [ ]:
np.eye(3)                   # identity matrix

In [ ]:
np.random.default_rng(42).random(3)   # 3 reproducible random floats in [0, 1)

### 1.3 · Operations: arithmetic, unary, ufuncs

Operations are *vectorised* — they apply to the whole array, no loop.

In [ ]:
# Month-on-month simple returns, the finance way
returns = (prices[1:] - prices[:-1]) / prices[:-1]
returns.round(4)

In [ ]:
prices.mean(), prices.max(), prices.std().round(2)     # unary aggregations

In [ ]:
np.sqrt(prices[:3])         # ufunc: element-wise. Others: np.exp, np.sin, np.cos

### 1.4 · Indexing & slicing

In [ ]:
prices[0], prices[-1], prices[2:4]   # first, last, a slice (stop is exclusive)

### 1.5 · Reshape & transpose

Same data, different shape — no copy of the values.

In [ ]:
mat = prices.reshape(2, 3)   # 2 rows x 3 cols
mat

In [ ]:
mat.T                        # transpose -> 3 x 2

### 1.6 · Stacking

In [ ]:
np.vstack([prices[:3], prices[3:]])   # stack rows; np.hstack joins along columns

### 1.7 · Copy vs. view

> **Note — a classic gotcha:** a slice is a *view* — it shares memory, so writing through it changes the original. `.copy()` breaks that link.

In [ ]:
v = prices[:2]      # view
v[0] = 0
prices[:2]          # original changed!

In [ ]:
prices[0] = 6800    # restore
c = prices[:2].copy()
c[0] = 0
prices[:2]          # original safe

### 1.8 · Broadcasting

A scalar (or smaller array) is "stretched" to match shape — no manual looping.

In [ ]:
prices * 1.10               # a 10% uniform uptick across all months

In [ ]:
prices - prices.mean()     # distance of each month from the average

---
# Part 2 · Pandas — Data Analysis

Pandas adds **labels** on top of NumPy. Two objects do almost everything: the 1-D **Series** (a labelled column) and the 2-D **DataFrame** (a labelled table).

### 2.1 · Series basics

A Series is values + an index.

In [ ]:
s = pd.Series(prices, index=["Jan", "Feb", "Mar", "Apr", "May", "Jun"], name="BAJFINANCE")
s

In [ ]:
s["Mar"], s.mean()         # label lookup and an aggregation

### 2.2 · DataFrame — our peer snapshot

The most common constructor: a dict of columns (illustrative figures).

In [ ]:
data = {
    "Ticker":   ["BAJFINANCE", "HDFCBANK", "KOTAKBANK", "SBIN", "ICICIBANK"],
    "Company":  ["Bajaj Finance", "HDFC Bank", "Kotak Mahindra", "State Bank", "ICICI Bank"],
    "Sector":   ["NBFC", "Bank", "Bank", "Bank", "Bank"],
    "CMP":      [7200, 1680, 1750, 820, 1150],          # current market price
    "MarketCapCr": [445000, 1280000, 348000, 732000, 808000],
    "PE":       [32.5, 19.2, 18.7, 9.8, 17.4],
    "ROE":      [21.8, 16.9, 14.2, 19.5, 18.1],         # %
}
df = pd.DataFrame(data)
df

### 2.3 · Other ways to build a DataFrame

Same table can come from a list of rows, or from arrays — handy when data arrives in different shapes.

In [ ]:
# From a list of tuples (rows) + explicit column names
rows = [("BAJFINANCE", 7200), ("HDFCBANK", 1680)]
pd.DataFrame(rows, columns=["Ticker", "CMP"])

### 2.4 · Attributes: shape, columns, dtypes, index

In [ ]:
df.shape, list(df.columns)

In [ ]:
df.dtypes

### 2.5 · head, tail, describe, info, sorting

In [ ]:
df.head(2)

In [ ]:
df.describe()              # summary stats for numeric columns

In [ ]:
df.sort_values("MarketCapCr", ascending=False)

### 2.6 · Selection: column, loc, iloc, boolean, slicing

> **Note:** `loc` selects by **label**, `iloc` by **integer position** — the most common point of confusion when you start out.

In [ ]:
df["Company"]              # a single column -> Series

In [ ]:
df.loc[0, "Company"], df.iloc[0, 1]   # label-based vs position-based

In [ ]:
df[df["PE"] < 20]         # boolean filter: 'value' names only

### 2.7 · Add / insert / delete columns

In [ ]:
df["MarketCapLakhCr"] = (df["MarketCapCr"] / 100000).round(2)   # add (computed)
df.insert(1, "Listed", "NSE")                                   # insert at position 1
df = df.drop(columns="Listed")                                  # delete
df.head(2)

### 2.8 · Cleaning: missing values, drop, replace, duplicates

Real data is messy. We inject a couple of gaps to show the tools.

In [ ]:
dirty = df.copy()
dirty.loc[2, "ROE"] = np.nan
dirty.loc[4, "ROE"] = np.nan
dirty[["Ticker", "ROE"]]

In [ ]:
dirty["ROE"].fillna(dirty["ROE"].mean()).round(2)   # fill gaps with the mean

In [ ]:
# ffill = carry last value forward; bfill = pull next value back
dirty["ROE"].ffill()

In [ ]:
df.replace({"NBFC": "Non-Bank"}).head(2)            # value replacement


In [ ]:
pd.concat([df, df.iloc[[0]]]).drop_duplicates().shape   # drop_duplicates removes the repeat

### 2.9 · Aggregation & GroupBy

In [ ]:
df.groupby("Sector")[["PE", "ROE"]].mean().round(2)   # bank vs NBFC averages

### 2.10 · Merge, concat, join, pivot

Combine tables that share a key.

In [ ]:
# A second table to merge in on Ticker
extra = pd.DataFrame({"Ticker": df["Ticker"], "NetProfitCr": [14450, 60800, 13780, 61080, 44256]})
merged = df.merge(extra, on="Ticker")
merged[["Ticker", "ROE", "NetProfitCr"]].head()

In [ ]:
pd.concat([df.head(2), df.tail(1)])   # stack rows from pieces

In [ ]:
df.pivot_table(index="Sector", values="ROE", aggfunc="mean").round(2)

### 2.11 · I/O: CSV & Excel

The everyday import/export. Excel needs the `openpyxl` engine installed.

In [ ]:
df.to_csv("peers.csv", index=False)
pd.read_csv("peers.csv").head(2)

In [ ]:
df.to_excel("peers.xlsx", index=False)
pd.read_excel("peers.xlsx").head(2)

### 2.12 · Basic visualization (Pandas shortcut)

Pandas wraps Matplotlib for one-line charts.

In [ ]:
df.plot(x="Ticker", y="MarketCapCr", kind="bar", legend=False,
        title="Market cap (Cr, illustrative)")
plt.tight_layout(); plt.show()

---
# Part 3 · Matplotlib — Data Visualization

For real control we drop to Matplotlib directly.

### 3.1 · The anatomy: Figure, Axes, Axis, Artist

- **Figure** — the whole canvas/page.
- **Axes** — one plot on it (you can have several). This is what you draw on.
- **Axis** — the x or y number line of an Axes.
- **Artist** — every visible thing (line, bar, label) is an Artist.

> **Note:** *Axes ≠ Axis.* The `fig, ax = plt.subplots()` pattern below is the modern, recommended way to start a chart.

In [ ]:
fig, ax = plt.subplots()
type(fig), type(ax)

### 3.2 · Subplots + titles, labels, legend, grid

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].plot(s.index, s.values, marker="o", label="BAJFINANCE")
axes[0].set_title("Price trend"); axes[0].set_xlabel("Month"); axes[0].set_ylabel("Price")
axes[0].legend(); axes[0].grid(True)
axes[1].bar(df["Ticker"], df["ROE"])
axes[1].set_title("ROE %"); axes[1].tick_params(axis="x", rotation=45)
plt.tight_layout(); plt.show()

### 3.3 · The core plot types

In [ ]:
# Line — a trend over time
plt.plot(s.index, s.values, marker="o", color="#1f77b4")
plt.title("BAJFINANCE month-end price"); plt.xlabel("Month"); plt.ylabel("Price")
plt.show()

In [ ]:
# Scatter — relationship between two numerics
plt.scatter(df["PE"], df["ROE"])
for _, r in df.iterrows():
    plt.annotate(r["Ticker"], (r["PE"], r["ROE"]), fontsize=8)
plt.xlabel("P/E"); plt.ylabel("ROE %"); plt.title("Valuation vs. return")
plt.show()

In [ ]:
# Bar — compare a value across categories
plt.bar(df["Ticker"], df["MarketCapCr"], color="#2ca02c")
plt.title("Market cap (Cr)"); plt.xticks(rotation=45); plt.show()

In [ ]:
# Histogram — distribution of 250 simulated daily returns (illustrative)
daily = np.random.default_rng(7).normal(0, 0.012, 250)
plt.hist(daily, bins=20, color="#9467bd")
plt.title("Daily return distribution"); plt.xlabel("Return"); plt.ylabel("Days")
plt.show()

In [ ]:
# Pie — share of a whole
plt.pie(df["MarketCapCr"], labels=df["Ticker"], autopct="%1.0f%%")
plt.title("Market-cap share"); plt.show()

### 3.4 · Styles, markers, colors & saving

In [ ]:
plt.plot(s.index, s.values, linestyle="--", marker="s", color="#d62728")
plt.title("Dashed line, square markers")
plt.savefig("bajfinance_price.png", dpi=120, bbox_inches="tight")   # export to file
plt.show()
print("saved -> bajfinance_price.png")

---
## Wrap-up

| Library | One-line role | We did |
|---|---|---|
| **NumPy** | fast typed arrays + vectorised math | prices, returns, reshape, broadcasting |
| **Pandas** | labelled tables, clean & combine | the peer table, selection, groupby, merge, I/O |
| **Matplotlib** | full control over charts | line, scatter, bar, histogram, pie, savefig |

> **Note:** every cell depends only on the cells above it, so always run the notebook top-to-bottom. Doing so writes three small files (`peers.csv`, `peers.xlsx`, `bajfinance_price.png`) into your folder — safe to delete afterwards.